In [1]:
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
import numpy as np
import pandas as pd
import yfinance as yf
import talib

C:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [2]:
cl = yf.Ticker("CL=F")
data = cl.history("max")

In [31]:
class StochMAStrategy(Strategy):
    ma_len = 66
    stoch_len = 14
    stoch_smooth = 2

    def init(self):
        close = self.data.Close

        # Trend filter
        self.ma = self.I(talib.MA, self.data.Close, self.ma_len)

        stoch_result = self.I(
            talib.STOCH, 
            self.data.High, 
            self.data.Low, 
            self.data.Close, 
            fastk_period=self.stoch_len,
            slowk_period=self.stoch_smooth,
            slowk_matype=0,
            slowd_period=3,
            slowd_matype=0
        )
        self.k = stoch_result[0]

    def next(self):
        price = self.data.Close[-1]
        trend_up = price > self.ma[-1]
        trend_down = price < self.ma[-1]
        
        # Buy signal: price above MA & %K crosses up above 35
        if trend_up and crossover(self.k, 35):
            self.position.close()
            self.buy(size=1)

        # Sell signal: price below MA & %K crosses down below 75
        if trend_down and crossover(75, self.k):
            self.position.close()
            self.sell(size=1)

In [32]:
bt = Backtest(data,StochMAStrategy,cash=10_000)

In [33]:
stats = bt.run()

Backtest.run:   0%|          | 0/6331 [00:00<?, ?bar/s]

C:\Users\DELL\AppData\Local\Temp\ipykernel_35708\2481999593.py:1: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = bt.run()


In [34]:
stats

Start                     2000-08-23 00:00...
End                       2026-02-13 00:00...
Duration                   9305 days 01:00:00
Exposure Time [%]                    97.23308
Equity Final [$]                   9787.45002
Equity Peak [$]                      10025.79
Return [%]                            -2.1255
Buy & Hold Return [%]                77.75579
Return (Ann.) [%]                     -0.0846
Volatility (Ann.) [%]                 0.28445
CAGR [%]                             -0.05817
Sharpe Ratio                         -0.29741
Sortino Ratio                        -0.41947
Calmar Ratio                         -0.03517
Alpha [%]                                 NaN
Beta                                      NaN
Max. Drawdown [%]                     -2.4055
Avg. Drawdown [%]                    -0.45861
Max. Drawdown Duration     6435 days 00:00:00
Avg. Drawdown Duration     1306 days 00:00:00
# Trades                                  233
Win Rate [%]                      

In [23]:
bt.plot()

GridPlot(id='p2268', ...)